# Nsquare | 이미지 분류 모델 학습 및 성능 개선

**CIFAR-10의 3개 클래스(plane / car / bird)** 를 분류하는 모델을 학습하고 평가합니다.

기본 CNN으로 먼저 전체 과정을 실행한 뒤, **모델 구조 · 모델 종류 · 학습 하이퍼파라미터**를 변경하여 성능을 비교합니다.

| 데이터 | 개수 | 용도 |
|---|---:|---|
| Training | 12,000장 | 모델 학습 (데이터 증강 적용 가능) |
| Validation | 3,000장 | 매 epoch 평가, 모델 및 하이퍼파라미터 선택 |
| Test | 3,000장 | 선택을 끝낸 후 **최종 1회 평가** |

**평가 Protocol:** CIFAR-10 클래스 0·1·2 사용, `SEED=42`로 원본 학습 이미지 15,000장을 12,000/3,000으로 고정 분할. 검증 정확도가 가장 높은 epoch의 가중치를 사용합니다. 정량 평가는 Accuracy·Confusion Matrix·클래스별 Precision/Recall/F1이며, 정성 평가는 실제 이미지와 예측 결과를 확인합니다. **Test는 튜닝에 사용하지 않습니다.**

*Google Colab에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하면 학습이 더 빠릅니다.*

## 1. 실행 환경

In [ ]:
import copy
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

SEED = 42
CLASS_NAMES = ["plane", "car", "bird"]
NUM_CLASSES = len(CLASS_NAMES)
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2023, 0.1994, 0.2010)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if device.type == "cuda" else 0
PIN_MEMORY = (device.type == "cuda")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU 모드입니다. Colab에서는 GPU 런타임을 권장합니다.")

## 2. 데이터 준비: Training / Validation / Test

In [ ]:
# 원본 노트북과 동일한 데이터 전처리
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 같은 CIFAR-10 학습 이미지라도 Training/Validation은 서로 다른 transform 사용
full_train_aug = datasets.CIFAR10(
    root="dataset/", train=True, download=True, transform=train_transform
)
full_train_plain = datasets.CIFAR10(
    root="dataset/", train=True, download=True, transform=eval_transform
)
full_test = datasets.CIFAR10(
    root="dataset/", train=False, download=True, transform=eval_transform
)

# 첫 세 클래스에 해당하는 이미지 인덱스만 추출
eligible_train_indices = [
    i for i, label in enumerate(full_train_aug.targets) if label < NUM_CLASSES
]
eligible_test_indices = [
    i for i, label in enumerate(full_test.targets) if label < NUM_CLASSES
]

# 원본과 완전히 동일한 split 로직 (바꾸지 않음)
split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(
    len(eligible_train_indices), generator=split_generator
).tolist()

train_indices = [eligible_train_indices[i] for i in permutation[:12000]]
val_indices = [eligible_train_indices[i] for i in permutation[12000:]]

train_dataset = Subset(full_train_aug, train_indices)
train_dataset_no_aug = Subset(full_train_plain, train_indices)
val_dataset = Subset(full_train_plain, val_indices)
test_dataset = Subset(full_test, eligible_test_indices)

assert (len(train_dataset), len(val_dataset), len(test_dataset)) == (12000, 3000, 3000)
assert set(train_indices).isdisjoint(set(val_indices)), "Train/Validation 중복이 있습니다."

# 실험 시 batch size와 training augmentation만 바꿀 수 있도록 DataLoader 분리
def make_loaders(batch_size=128, augmentation=True):
    selected_train_dataset = train_dataset if augmentation else train_dataset_no_aug
    train_loader = DataLoader(
        selected_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=256, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    return train_loader, val_loader

# 최종 Test용 Loader: 실험/학습 함수에 넘기지 않음
test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

print(f"Training:   {len(train_dataset):,} images")
print(f"Validation: {len(val_dataset):,} images")
print(f"Test:       {len(test_dataset):,} images")

In [ ]:
# 이미지와 클래스 분포 확인
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), np.random.default_rng(SEED).choice(len(val_dataset), 10, replace=False)):
    image, label = val_dataset[int(idx)]
    image = image.permute(1, 2, 0).numpy()
    image = np.clip(image * np.array(STD) + np.array(MEAN), 0, 1)
    ax.imshow(image)
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
plt.suptitle("Validation images (no augmentation)")
plt.tight_layout()
plt.show()

for title, source, indices in [
    ("Train", full_train_plain, train_indices),
    ("Validation", full_train_plain, val_indices),
    ("Test", full_test, eligible_test_indices),
]:
    counts = np.bincount([source.targets[i] for i in indices], minlength=NUM_CLASSES)
    print(f"{title:10s} : " + "  ".join(f"{name}={count:,}" for name, count in zip(CLASS_NAMES, counts)))

## 3. 모델 정의

아래의 `build_model()`에서 모델을 선택합니다. **사전학습 가중치 없이(`weights=None`)** 동일한 3개 클래스에 대해 학습합니다.

- `cnn`: 직접 구성한 CNN. `cnn_width`, `cnn_blocks`, `dropout`으로 구조 변경
- `resnet18`, `resnet34`, `resnet50`: CIFAR-10 크기에 맞게 첫 convolution을 `3×3 / stride 1`로 변경하고 첫 max-pooling 제거
- `vgg16_bn`: VGG16의 13개 convolution 및 BatchNorm을 사용하고 출력층을 3클래스로 변경

In [ ]:
class SimpleCNN(nn.Module):
    """너비와 블록 수를 조정할 수 있는 소형 CNN."""
    def __init__(self, width=32, num_blocks=3, dropout=0.10, num_classes=NUM_CLASSES):
        super().__init__()
        if num_blocks not in (2, 3, 4):
            raise ValueError("cnn_blocks는 2, 3, 4 중 선택하세요.")
        if not (0.0 <= dropout < 1.0):
            raise ValueError("dropout은 0 이상 1 미만이어야 합니다.")

        layers = []
        in_channels = 3
        for block_idx in range(num_blocks):
            out_channels = width * (2 ** block_idx)
            for _ in range(2):
                layers.extend([
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True),
                ])
                in_channels = out_channels
            layers.append(nn.MaxPool2d(kernel_size=2))
            if dropout > 0:
                layers.append(nn.Dropout2d(p=dropout))

        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(in_channels, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(config):
    name = config["model"].lower()
    if name == "cnn":
        return SimpleCNN(
            width=config.get("cnn_width", 32),
            num_blocks=config.get("cnn_blocks", 3),
            dropout=config.get("dropout", 0.10),
        )

    if name in ("resnet18", "resnet34", "resnet50"):
        model = getattr(models, name)(weights=None)
        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        model.maxpool = nn.Identity()
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        return model

    if name == "vgg16_bn":
        model = models.vgg16_bn(weights=None)
        model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        model.classifier = nn.Linear(512, NUM_CLASSES)
        return model

    raise ValueError("model은 cnn, resnet18, resnet34, resnet50, vgg16_bn 중 선택하세요.")


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 4. 학습 및 검증 함수

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return {"loss": running_loss / total, "accuracy": 100.0 * correct / total}


@torch.inference_mode()
def evaluate(model, loader, criterion, collect_predictions=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_true, all_pred, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        outputs = model(images)
        loss = criterion(outputs, labels)
        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (predictions == labels).sum().item()
        total += batch_size

        if collect_predictions:
            all_true.append(labels.cpu().numpy())
            all_pred.append(predictions.cpu().numpy())
            all_probs.append(torch.softmax(outputs, dim=1).cpu().numpy())

    result = {"loss": running_loss / total, "accuracy": 100.0 * correct / total}
    if collect_predictions:
        result["y_true"] = np.concatenate(all_true)
        result["y_pred"] = np.concatenate(all_pred)
        result["probabilities"] = np.concatenate(all_probs)
    return result


def make_optimizer(model, config):
    name = config["optimizer"].lower()
    if name == "adam":
        return optim.Adam(
            model.parameters(), lr=config["lr"],
            weight_decay=config["weight_decay"],
        )
    if name == "sgd":
        return optim.SGD(
            model.parameters(), lr=config["lr"],
            momentum=config.get("momentum", 0.9),
            weight_decay=config["weight_decay"],
        )
    if name == "adamw":
        return optim.AdamW(
            model.parameters(), lr=config["lr"],
            weight_decay=config["weight_decay"],
        )
    raise ValueError("optimizer는 adam, adamw, sgd 중 선택하세요.")


def make_scheduler(optimizer, config):
    name = config["scheduler"].lower()
    if name == "none":
        return None
    if name == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
    if name == "step":
        return optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, config["epochs"] // 3), gamma=0.5
        )
    raise ValueError("scheduler는 none, cosine, step 중 선택하세요.")


def train_experiment(name, config):
    """Validation accuracy로 최적 epoch 선택; Test 데이터는 사용하지 않음."""
    if not (1 <= config["epochs"] <= 15):
        raise ValueError("실습에서 epochs는 1~15로 설정합니다.")
    if config["batch_size"] < 1 or config["lr"] <= 0:
        raise ValueError("batch_size는 양의 정수, lr은 양수여야 합니다.")

    set_seed(SEED)  # 매 실험은 같은 seed로 시작
    train_loader, val_loader = make_loaders(
        batch_size=config["batch_size"],
        augmentation=config["augmentation"],
    )
    model = build_model(config).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, config)
    scheduler = make_scheduler(optimizer, config)

    history = []
    best_val_accuracy = -1.0
    best_epoch = None
    best_model_state = None
    start = time.time()

    print(f"\n[{name}] model={config['model']} / epochs={config['epochs']} / "
          f"batch={config['batch_size']} / lr={config['lr']} / optimizer={config['optimizer']}")
    print(f"Trainable parameters: {count_parameters(model):,}")

    for epoch in range(1, config["epochs"] + 1):
        lr_now = optimizer.param_groups[0]["lr"]
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = evaluate(model, val_loader, criterion)

        if val_metrics["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_metrics["accuracy"]
            best_epoch = epoch
            best_model_state = copy.deepcopy({
                key: value.detach().cpu() for key, value in model.state_dict().items()
            })

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "lr": lr_now,
        })
        print(
            f"Epoch {epoch:02d}/{config['epochs']} | "
            f"Train loss {train_metrics['loss']:.4f} / acc {train_metrics['accuracy']:.2f}% | "
            f"Val loss {val_metrics['loss']:.4f} / acc {val_metrics['accuracy']:.2f}%"
        )
        if scheduler is not None:
            scheduler.step()

    # 실험별 베스트 가중치를 디스크에 저장(메모리 누적 방지)
    checkpoint_path = CHECKPOINT_DIR / f"{name}.pth"
    torch.save(best_model_state, checkpoint_path)
    result = {
        "name": name,
        "config": copy.deepcopy(config),
        "best_val_accuracy": best_val_accuracy,
        "best_epoch": best_epoch,
        "checkpoint": str(checkpoint_path),
        "params": count_parameters(model),
        "seconds": time.time() - start,
        "history": pd.DataFrame(history),
    }
    print(f"Best validation accuracy: {best_val_accuracy:.2f}% "
          f"(epoch {best_epoch}) / 시간: {result['seconds']:.1f}초")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result


def load_experiment_model(result):
    model = build_model(result["config"])
    state = torch.load(result["checkpoint"], map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    return model.to(device).eval()

In [ ]:
# 평가 결과를 그리는 함수들

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="Train")
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="Validation")
    axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
    axes[0].legend()

    axes[1].plot(history["epoch"], history["train_accuracy"], marker="o", label="Train")
    axes[1].plot(history["epoch"], history["val_accuracy"], marker="o", label="Validation")
    axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    axes[1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def show_quantitative_results(metrics, title):
    print(f"{title} Accuracy: {metrics['accuracy']:.2f}%")
    print(f"{title} Cross-entropy loss: {metrics['loss']:.4f}")
    print("\nClassification report (Precision / Recall / F1)")
    print(classification_report(
        metrics["y_true"], metrics["y_pred"],
        labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES,
        digits=3, zero_division=0,
    ))
    cm = confusion_matrix(
        metrics["y_true"], metrics["y_pred"], labels=list(range(NUM_CLASSES))
    )
    print("Confusion Matrix (rows=actual, columns=predicted)")
    print(cm)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(f"{title} Confusion Matrix")
    fig.tight_layout()
    plt.show()


def show_qualitative_results(metrics, dataset, title, n_wrong=8, n_correct=4):
    """이미 계산한 예측값으로 성공/오분류 이미지를 표시 (추가 모델 평가 없음)."""
    truth = metrics["y_true"]
    pred = metrics["y_pred"]
    probabilities = metrics["probabilities"]
    wrong = np.flatnonzero(truth != pred)
    correct = np.flatnonzero(truth == pred)
    rng = np.random.default_rng(SEED)

    wrong_pick = rng.choice(wrong, size=min(n_wrong, len(wrong)), replace=False)
    correct_pick = rng.choice(correct, size=min(n_correct, len(correct)), replace=False)
    chosen = np.concatenate([wrong_pick, correct_pick]).astype(int)
    if len(chosen) == 0:
        print("표시할 예측이 없습니다.")
        return

    columns = 4
    rows = int(np.ceil(len(chosen) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, rows * 3))
    axes = np.atleast_1d(axes).ravel()
    for ax, idx in zip(axes, chosen):
        image, true_label = dataset[idx]
        image = image.permute(1, 2, 0).numpy()
        image = np.clip(image * np.array(STD) + np.array(MEAN), 0, 1)
        predicted_label = int(pred[idx])
        confidence = float(probabilities[idx, predicted_label]) * 100.0
        correct_mark = "✓" if true_label == predicted_label else "✗"

        ax.imshow(image)
        ax.set_title(
            f"{correct_mark} True: {CLASS_NAMES[true_label]}\n"
            f"Pred: {CLASS_NAMES[predicted_label]} ({confidence:.1f}%)",
            fontsize=10,
        )
        ax.axis("off")
    for ax in axes[len(chosen):]:
        ax.axis("off")
    fig.suptitle(f"{title}: misclassified {len(wrong_pick)} + correct {len(correct_pick)}")
    fig.tight_layout()
    plt.show()

## 5. Baseline 학습: 순서대로 실행하면 완료

`BASELINE_CONFIG`에서 실습 시작 모델과 학습 설정을 지정합니다. 기본값은 **소형 CNN / 8 epochs**이며, 최적 모델은 **Validation Accuracy**로 자동 선택됩니다. Test 데이터는 이 단계에서 보지 않습니다.

In [ ]:
BASELINE_CONFIG = {
    "model": "cnn",           # cnn / resnet18 / resnet34 / resnet50 / vgg16_bn
    "cnn_width": 32,          # CNN 첫 블록 채널 수 (CNN에만 적용)
    "cnn_blocks": 3,         # CNN 블록 개수: 2, 3, 4 (CNN에만 적용)
    "dropout": 0.10,          # CNN dropout 비율 (CNN에만 적용)
    "epochs": 8,             # 1~15
    "batch_size": 128,
    "lr": 0.001,
    "optimizer": "adam",      # adam / adamw / sgd
    "momentum": 0.9,         # SGD에서 사용
    "weight_decay": 0.0001,
    "scheduler": "cosine",    # cosine / step / none
    "augmentation": True,    # 학습 데이터에만 증강 적용
}

baseline_result = train_experiment("baseline", BASELINE_CONFIG)

In [ ]:
# 정량 평가 ①: Epoch별 학습/검증 추이
plot_history(baseline_result["history"], "Baseline: training / validation")
print("Best Validation Accuracy:", f"{baseline_result['best_val_accuracy']:.2f}%")
print("Best Epoch:", baseline_result["best_epoch"])

## 6. Baseline 평가: 정량 + 정성 (Validation)

In [ ]:
# 정량 평가 ②: Accuracy, Precision/Recall/F1, Confusion Matrix
# 정성 평가: 정답을 맞힌 이미지와 잘못 분류한 이미지 확인
baseline_model = load_experiment_model(baseline_result)
_, validation_loader = make_loaders(
    batch_size=BASELINE_CONFIG["batch_size"], augmentation=BASELINE_CONFIG["augmentation"]
)
criterion = nn.CrossEntropyLoss()
baseline_val_metrics = evaluate(
    baseline_model, validation_loader, criterion, collect_predictions=True
)
show_quantitative_results(baseline_val_metrics, "Baseline Validation")
show_qualitative_results(baseline_val_metrics, val_dataset, "Baseline Validation")
del baseline_model
if device.type == "cuda":
    torch.cuda.empty_cache()

## 7. 성능 개선 실습: 바꿔 보고 비교하기

**변경 예시**

| 실습 방향 | 바꿔 볼 설정 | 예시 |
|---|---|---|
| CNN 구조 변경 | `cnn_width`, `cnn_blocks`, `dropout` | 너비 32 → 64, 블록 3 → 4 |
| 모델 종류 변경 | `model` | CNN → ResNet18/34/50 또는 VGG16 |
| 학습 하이퍼파라미터 | `lr`, `optimizer`, `scheduler`, `batch_size` | Adam → SGD, learning rate 변경 |
| 데이터 증강 | `augmentation` | RandomCrop/Flip 적용 여부 |

아래 사전 정의된 실험을 골라 실행하거나, `EXPERIMENT_PRESETS`에 새 설정을 추가할 수 있습니다. **실험 모델을 비교하고 최종 모델을 선택할 때는 Validation Accuracy만 사용합니다.** ResNet50/VGG16은 학습 시간이 더 걸릴 수 있습니다.

In [ ]:
# BASELINE_CONFIG에서 달라지는 부분만 작성하면 됩니다.
EXPERIMENT_PRESETS = {
    # (1) 모델 구조 변경
    "wider_cnn":  {"model": "cnn", "cnn_width": 64},
    "deeper_cnn": {"model": "cnn", "cnn_blocks": 4},
    "less_dropout": {"model": "cnn", "dropout": 0.0},

    # (2) 모델 종류 변경: pretrained weights 사용하지 않음
    "resnet18": {"model": "resnet18", "epochs": 10, "batch_size": 128},
    "resnet34": {"model": "resnet34", "epochs": 10, "batch_size": 128},
    "resnet50": {"model": "resnet50", "epochs": 10, "batch_size": 64,
                 "lr": 0.0003},
    "vgg16_bn": {"model": "vgg16_bn", "epochs": 10, "batch_size": 64},

    # (3) 학습 방법 및 하이퍼파라미터 변경
    "smaller_lr": {"lr": 0.0003},
    "sgd_cosine": {"optimizer": "sgd", "lr": 0.05, "momentum": 0.9,
                   "scheduler": "cosine"},
    "larger_batch": {"batch_size": 256},
    "no_augmentation": {"augmentation": False},

    # (4) 직접 구성한 실험 예시
    "my_experiment": {"model": "cnn", "cnn_width": 48, "lr": 0.0005},
}

# 처음 [런타임 → 모두 실행] 할 때는 False: Baseline 학습과 최종 Test까지 자동 실행
# 추가 실습 시 True로 바꾸고 SELECTED_EXPERIMENTS에서 이름을 선택해 다시 실행
RUN_EXTRA_EXPERIMENTS = False
SELECTED_EXPERIMENTS = ["wider_cnn", "resnet18"]

experiment_results = [baseline_result]
if RUN_EXTRA_EXPERIMENTS:
    for experiment_name in SELECTED_EXPERIMENTS:
        if experiment_name not in EXPERIMENT_PRESETS:
            raise KeyError(f"없는 실험: {experiment_name}")
        updated_config = {**BASELINE_CONFIG, **EXPERIMENT_PRESETS[experiment_name]}
        result = train_experiment(experiment_name, updated_config)
        experiment_results.append(result)
else:
    print("추가 실험은 건너뜁니다. RUN_EXTRA_EXPERIMENTS=True로 바꾸면 선택한 실험들이 실행됩니다.")

In [ ]:
# 실험 비교: Test 결과가 아니라 Validation 성능으로 비교합니다.
comparison = pd.DataFrame([
    {
        "experiment": r["name"],
        "model": r["config"]["model"],
        "best_epoch": r["best_epoch"],
        "best_val_accuracy (%)": round(r["best_val_accuracy"], 2),
        "parameters": r["params"],
        "seconds": round(r["seconds"], 1),
    }
    for r in experiment_results
]).sort_values("best_val_accuracy (%)", ascending=False).reset_index(drop=True)

display(comparison)

# 반올림하지 않은 Validation Accuracy로 최종 후보 선택
best_experiment = max(experiment_results, key=lambda r: r["best_val_accuracy"])
print("Selected model (validation only):", best_experiment["name"])
print("Best Validation Accuracy:", f"{best_experiment['best_val_accuracy']:.2f}%")

if len(comparison) > 1:
    fig, ax = plt.subplots(figsize=(max(7, len(comparison) * 1.5), 4))
    ax.bar(comparison["experiment"], comparison["best_val_accuracy (%)"])
    ax.set_ylabel("Best validation accuracy (%)")
    ax.set_title("Experiment comparison (validation only)")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()
    plot_history(best_experiment["history"], f"Selected: {best_experiment['name']}")

In [ ]:
# 추가 실험으로 선택 모델이 바뀌었다면, Test를 열어보기 전에 Validation에서 정량·정성 비교
if best_experiment["name"] != "baseline":
    selected_model = load_experiment_model(best_experiment)
    _, selected_val_loader = make_loaders(
        batch_size=best_experiment["config"]["batch_size"],
        augmentation=best_experiment["config"]["augmentation"],
    )
    selected_val_metrics = evaluate(
        selected_model, selected_val_loader, nn.CrossEntropyLoss(),
        collect_predictions=True,
    )
    show_quantitative_results(selected_val_metrics, "Selected Model Validation")
    show_qualitative_results(selected_val_metrics, val_dataset, "Selected Model Validation")
    del selected_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("현재 선택된 모델은 Baseline입니다. 위 Validation 평가를 참고하세요.")

## 8. 최종 평가: Test (1회)

학습 및 하이퍼파라미터 비교를 마친 후, **Validation Accuracy가 가장 높은 실험의 최적 가중치**로 Test 3,000장을 평가합니다. Test의 Accuracy·Confusion Matrix·Precision/Recall/F1을 확인하고, 정분류/오분류 이미지를 살펴봅니다.

**이 결과를 확인한 뒤 Test 성능을 기준으로 다시 튜닝하지 않습니다.** 실험을 처음부터 변경할 때는 새 Colab 런타임에서 검증 데이터 기준으로 다시 비교하고, 최종 Test는 마지막에 한 번만 실행합니다.

In [ ]:
# 최종 Test 실행은 명시적으로 1회만 허용 (같은 런타임에서 중복 평가 방지)
if globals().get("FINAL_TEST_EVALUATED", False):
    raise RuntimeError("이 런타임에서 Test 평가를 이미 수행했습니다. 결과는 위 셀을 확인하세요.")

final_model = load_experiment_model(best_experiment)
final_test_metrics = evaluate(
    final_model, test_loader, nn.CrossEntropyLoss(), collect_predictions=True
)
FINAL_TEST_EVALUATED = True

# 선택된 최종 가중치 저장 (test 예측으로 가중치를 변경하지 않음)
final_model_path = "Nsquare_selected_best.pth"
torch.save({
    "model_state_dict": torch.load(
        best_experiment["checkpoint"], map_location="cpu", weights_only=True
    ),
    "model_config": best_experiment["config"],
    "selected_experiment": best_experiment["name"],
    "best_val_accuracy": best_experiment["best_val_accuracy"],
    "best_epoch": best_experiment["best_epoch"],
}, final_model_path)

print("=" * 60)
print("FINAL TEST EVALUATION")
print("=" * 60)
print(f"Selected model       : {best_experiment['name']} ({best_experiment['config']['model']})")
print(f"Best validation acc  : {best_experiment['best_val_accuracy']:.2f}%")
print(f"Final test accuracy  : {final_test_metrics['accuracy']:.2f}%")
print(f"Checkpoint           : {final_model_path}")
print("=" * 60)

# 정량적 평가
show_quantitative_results(final_test_metrics, "Final Test")

# 정성적 평가 (위에서 만든 예측값만 이용; Test를 다시 추론하지 않음)
show_qualitative_results(final_test_metrics, test_dataset, "Final Test")